In [ ]:
import pandas as pd
from datetime import datetime

import src.data_preparation.data_preprocessing.processing_helpers as processing
import src.data_preparation.data_collection.data_query as query
from src.data_preparation.data_collection.databases import player_game_database, player_info_database, team_game_database
from src.data_preparation.components.nba_data import NBAData
import src.data_preparation.data_preprocessing.player_props_processing as player_props
import train

In [ ]:
pd.set_option("display.max_rows", 10)
pd.set_option("display.max_columns", 100)

# PLAYER GAME DATA: #

In [ ]:
player_game_data = query.get_all_player_game_data()
player_game_data = player_game_data.dropna()
player_game_data

In [ ]:
player_game_data = processing.add_points_column(
    player_game_data,
    player_game_database.FIELD_GOALS_MADE.category_name, 
    player_game_database.THREE_POINTERS_MADE.category_name, 
    player_game_database.FREE_THROWS_MADE.category_name
    )
player_game_data

In [ ]:
player_game_data = player_game_data.sort_values(by=player_game_database.GAME_DATE.category_name)
player_game_data

In [ ]:
player_game_data =processing.add_rest_column(
    player_game_data,
    player_game_database.GAME_DATE.category_name
    )
player_game_data

In [ ]:
player_game_data = processing.add_total_rebounds_column(
    player_game_data,
    player_game_database.OFFENSIVE_REBOUNDS.category_name,
    player_game_database.DEFENSIVE_REBOUNDS.category_name    
    )
player_game_data

In [ ]:
new_dict = {}
for player_id, game_data in player_game_data.groupby(by=player_game_database.PLAYER_ID.category_name):
    sorted_data = game_data.sort_values(by=player_game_database.GAME_DATE.category_name)
    new_dict[player_id] = processing.add_rest_column(
    sorted_data,
    player_game_database.GAME_DATE.category_name
    )

new_dict

# TEAM GAME DATA: #

In [ ]:
team_game_data = query.get_all_team_game_data()
team_game_data = team_game_data.dropna()
team_game_data

In [ ]:
team_game_data = team_game_data.sort_values(by=team_game_database.GAME_DATE.category_name)
team_game_data

In [ ]:
team_game_data = processing.add_encoded_home_or_away_column(
    team_game_data,
    team_game_database.MATCHUP.category_name
    )
team_game_data

In [ ]:
team_game_data = processing.add_possessions_column(
    team_game_data,
    team_game_database.FIELD_GOALS_ATTEMPTED.category_name,
    team_game_database.FREE_THROWS_ATTEMPTED.category_name,
    team_game_database.OFFENSIVE_REBOUNDS.category_name,
    team_game_database.TURNOVERS.category_name    
    )
team_game_data

In [ ]:
team_game_data = processing.add_encoded_season_type_columns(
    team_game_data,
    team_game_database.GAME_ID.category_name
    )
team_game_data

In [ ]:
team_game_data = processing.add_opponent_stat_columns(
    team_game_data,
    team_game_database.GAME_ID.category_name,
    team_game_database.TEAM_ID.category_name,
    )
team_game_data

# PLAYER INFO: #

In [ ]:
player_info = query.get_all_players_info()
player_info = player_info.dropna()
player_info

In [ ]:
player_info['NAME'] = player_info[player_info_database.PLAYER_ID.category_name].apply(lambda x : NBAData.player_id_to_name(x))
player_info

In [ ]:
na_data = player_info[player_info[player_info_database.HEIGHT.category_name].isna()]

In [ ]:
missing_data = player_info[
    (player_info[player_info_database.HEIGHT.category_name].astype(str) == '') |
    (player_info[player_info_database.WEIGHT.category_name].astype(str) == '') |
    (player_info[player_info_database.POSITION.category_name].astype(str) == '') | 
    (player_info[player_info_database.FROM_YEAR.category_name].astype(str) == '') | 
    (player_info[player_info_database.TO_YEAR.category_name].astype(str) == '')
]
missing_data

In [ ]:
missing_player_ids = set(missing_data[player_info_database.PLAYER_ID.category_name])

games_with_missing_player_info = player_game_data[
    player_game_data[player_game_database.PLAYER_ID.category_name].isin(
        missing_player_ids
    )
    ]

games_with_missing_player_info

In [ ]:
invalid_weights = player_info[
    player_info[player_info_database.WEIGHT.category_name].astype(str) == ''
]
invalid_weights

In [ ]:
player_info = processing.convert_height_to_inches(
    player_info,
    player_info_database.HEIGHT.category_name
)
player_info

In [ ]:
positions = set(player_info[player_info_database.POSITION.category_name])
positions

In [ ]:
player_info[player_info_database.BIRTHDATE.category_name] = pd.to_datetime(player_info[player_info_database.BIRTHDATE.category_name])
player_info

In [ ]:
player_info[player_info_database.FROM_YEAR.category_name] = pd.to_datetime(player_info[player_info_database.FROM_YEAR.category_name], format='%Y')
player_info

# PLAYER PROPS:

In [ ]:
datasets = player_props.get_all_player_props_input_output_training_data(
        validation_split_start_date = datetime(2024, 1, 1),
        test_split_start_date = datetime(2025, 1, 1),
        chunk_step_size = 10,
        print_debug = True
        )
datasets

In [ ]:
train.squisher_and_cleaner(
    datasets[0]
)